In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [ ]:
df = pd.read_csv("/content/glass.csv")

In [ ]:
print("Shape:", df.shape)
print("Columns:", df.columns)
print(df.head())

Shape: (214, 10)
Columns: Index(['RI', 'Na', 'Mg', 'Al', 'Si', 'K', 'Ca', 'Ba', 'Fe', 'Type'], dtype='object')
        RI     Na    Mg    Al     Si     K    Ca   Ba   Fe  Type
0  1.52101  13.64  4.49  1.10  71.78  0.06  8.75  0.0  0.0     1
1  1.51761  13.89  3.60  1.36  72.73  0.48  7.83  0.0  0.0     1
2  1.51618  13.53  3.55  1.54  72.99  0.39  7.78  0.0  0.0     1
3  1.51766  13.21  3.69  1.29  72.61  0.57  8.22  0.0  0.0     1
4  1.51742  13.27  3.62  1.24  73.08  0.55  8.07  0.0  0.0     1


This step loads the dataset into a table (DataFrame).
Each row represents one glass sample and each column represents a feature or label.
The model can only work with numeric data, so loading correctly is important.

In [ ]:
df["y"] = (df["Type"] == 1).astype(int)
df = df.drop(columns=["Type"])

X = df.drop(columns=["y"]).values
y = df["y"].values

The original dataset has multiple glass types.
We convert it into a binary problem (Type 1 vs Others).
This simplifies the task into YES or NO classification.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

The dataset is divided into training and testing sets.
The model learns from training data and is evaluated on unseen test data.
This prevents false confidence.

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


Features have different numeric ranges.
Scaling ensures all features contribute equally.
Without scaling, learning becomes unstable.



In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

Sigmoid converts the score into a probability between 0 and 1.
Unlike perceptron, it shows confidence instead of just YES or NO.

In [ ]:
def predict_proba(X, w, b):
    z = np.dot(X, w) + b
    p = sigmoid(z)
    return p


In [ ]:
def loss(y, p):
    epsilon = 1e-9  # avoid log(0)
    return -np.mean(y*np.log(p+epsilon) + (1-y)*np.log(1-p+epsilon))


Binary cross-entropy measures how wrong the model’s confidence is.
Confident wrong predictions are penalized more.

In [ ]:
def update_weights(X, y, w, b, lr):
    p = predict_proba(X, w, b)
    error = p - y

    w = w - lr * np.dot(X.T, error) / len(y)
    b = b - lr * np.mean(error)

    return w, b


In [ ]:
w = np.zeros(X_train.shape[1])
b = 0.0
lr = 0.1
epochs = 100

for epoch in range(epochs):
    w, b = update_weights(X_train, y_train, w, b, lr)

    if epoch % 10 == 0:
        p_train = predict_proba(X_train, w, b)
        print(f"Epoch {epoch} | Loss: {loss(y_train, p_train):.4f}")

Epoch 0 | Loss: 0.6822
Epoch 10 | Loss: 0.6107
Epoch 20 | Loss: 0.5748
Epoch 30 | Loss: 0.5529
Epoch 40 | Loss: 0.5379
Epoch 50 | Loss: 0.5269
Epoch 60 | Loss: 0.5184
Epoch 70 | Loss: 0.5116
Epoch 80 | Loss: 0.5060
Epoch 90 | Loss: 0.5014


During training, weights and bias are updated to reduce loss.
Over epochs, predictions become more accurate and meaningful.

In [ ]:
def predict_label(p, threshold=0.5):
    return (p >= threshold).astype(int)

# Evaluate with threshold 0.5
p_test = predict_proba(X_test, w, b)
y_pred_05 = predict_label(p_test, threshold=0.5)
print("\nAccuracy (0.5 threshold):", accuracy_score(y_test, y_pred_05))

# Evaluate with threshold 0.7
y_pred_07 = predict_label(p_test, threshold=0.7)
print("Accuracy (0.7 threshold):", accuracy_score(y_test, y_pred_07))


Accuracy (0.5 threshold): 0.8604651162790697
Accuracy (0.7 threshold): 0.7209302325581395


The threshold converts probability into a final decision.
A higher threshold (0.7) is safer because it reduces false positives in glass quality control.

Logistic regression differs from perceptron because it outputs probabilities instead of hard decisions.
The sigmoid function allows smooth learning using cross-entropy loss.
However, the model is still linear and cannot capture complex nonlinear patterns.